### Sprint 7 – Relatórios

In [1]:
# Importar bibliotecas

import pandas as pd
import joblib
from pathlib import Path

In [2]:
# Diretório raiz do projeto

ROOT_DIR = Path.cwd().parent

# Diretórios do projeto

DATA_RAW_DIR = ROOT_DIR / "data" / "raw"
DATA_PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODEL_DIR = ROOT_DIR / "models"
OUTPUT_DIR = ROOT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Importar o modelo

modelo = joblib.load(MODEL_DIR / "xgb_turnover.pkl")

In [4]:
# Importar o conjunto de teste

X_test = pd.read_csv(DATA_PROCESSED_DIR / "X_test.csv")

X_test.shape

(294, 44)

In [5]:
# Gerar probabilidades

probabilidades = modelo.predict_proba(X_test)[:, 1]
probabilidades[:10]

array([9.0984106e-01, 5.6521915e-04, 1.5254115e-02, 4.5269902e-05,
       4.4096843e-03, 4.9242113e-02, 2.5436650e-03, 5.4549472e-04,
       2.6599839e-04, 6.2885785e-01], dtype=float32)

In [6]:
# Criar DataFrame com as probabilidades

ranking = pd.DataFrame({
    "Probabilidade": probabilidades
})

In [7]:
# Criar classificação

def classificar(prob):

    if prob >= 0.70:
        return "Alto"

    elif prob >= 0.40:
        return "Médio"

    else:
        return "Baixo"
    
    ranking["Risco"] = ranking["Probabilidade"].apply(classificar)

In [8]:
# Converter em porcentagem

ranking["Probabilidade (%)"] = (
    ranking["Probabilidade"] * 100
).round(2)

ranking = ranking.drop(columns="Probabilidade")

In [9]:
# Ordenar

ranking = ranking.sort_values(
    by="Probabilidade (%)",
    ascending=False
)

In [10]:
# Visualizar DF

ranking.head(20)

,Probabilidade (%)
200,99.989998
92,99.940002
276,99.529999
81,99.099998
287,98.470001
214,98.349998
251,97.300003
223,95.910004
35,95.489998
199,94.879997


In [11]:
# Definir cabeçalho para ID

ranking = ranking.reset_index()
ranking = ranking.rename(columns={"index": "ID"})

In [12]:
# Exportar ranking para Excel

ranking.to_excel(
    OUTPUT_DIR / "turnover_risk_ranking.xlsx",
    index=False
)